# Step 2 - Feature Construction: Genomic + Clinical Predictors

Builds the predictor matrix (BASE cohort, n=10,000) from genomic + clinical data only

## Genomic feature methodology note (important caveat)

While building the APOE feature, the two APOE-defining SNPs (`rs429358`, `rs7412`) were found at positions far apart with allele codes that don't match their real GRCh37 coordinates/alleles (real: ~138bp apart at chr19:45.41M, C/T alleles both). **This SNP array's position and allele fields are synthetic/scrambled and cannot be trusted to reflect real genome coordinates**, even for these two unambiguous, famous SNPs. Consequently:

- **PRS cannot be built by matching array variants to a real published GWAS's summary statistics** (neither by position nor with confidence in allele-direction concordance for the few real rsIDs present).
- Instead, PRS here is **derived in-sample**: a genome-wide association scan of each cardiometabolic diagnosis (`T_HTN`, `T_DM`, `T_LIP`) against all ~1M array SNPs, using 5-fold out-of-fold discovery/scoring so no individual's PRS is derived from a GWAS that included their own diagnosis. See `prs_pipeline/build_prs_pipeline.sh` for the full reproducible pipeline (already run; this notebook loads its outputs).
- **APOE** is similarly treated as a genotype-dosage feature at `rs429358`/`rs7412` rather than a validated clinical e2/e3/e4 readout, since real-world allele-to-isoform mapping isn't verifiable here.

This was confirmed with the user as the intended approach given the synthetic data's constraints.

In [ ]:
import numpy as np
import pandas as pd

CLINICAL_DIR = "/mnt/d/Healthcare_AI_Challenge/Healthcare_AI_Challenge_2026/CLINICAL_DATA"
STEP_DIR = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Notebook/20260930_Step1"
PRS_DISEASES = {  # PRS feature name -> (genomics directory, diagnosis it was derived from; 1=no, 2=yes)
    "PRS_HTN": ("genomics_htn", "T_HTN"),
    "PRS_DM": ("genomics_dm", "T_DM"),
    "PRS_LIP": ("genomics_lip", "T_LIP"),
}

TRUE_MISSING = [66666, 88888, 99999]  # not surveyed / - / unknown-no response
NOT_APPLICABLE = 77777  # skip pattern (e.g. never smoked -> smoking amount not asked)

data1 = pd.read_csv(f"{CLINICAL_DIR}/BASE_DATA1.txt")
data2 = pd.read_csv(f"{CLINICAL_DIR}/BASE_DATA2.txt")
data3 = pd.read_csv(f"{CLINICAL_DIR}/BASE_DATA3.txt")

clinical = data1.merge(data2, on=["T_ID", "T_DATA_CLASS"]).merge(data3, on=["T_ID", "T_DATA_CLASS"])
print(clinical.shape)
clinical = clinical.replace(TRUE_MISSING, np.nan)

# 77777 on exposure amount/duration fields means no exposure, i.e. 0 - not missing.
# (Previously all sentinels -> NaN -> median-imputed, which gave never-smokers the median pack-years.)
for col in ["T_SMDUYR", "T_SMAM", "T_DRDU", "T_PACKYR", "T_TOTALC"]:
    clinical[col] = clinical[col].replace(NOT_APPLICABLE, 0)
# remaining 77777 (disease-onset ages of non-diseased, sex-specific variables) -> NaN, handled below
clinical = clinical.replace(NOT_APPLICABLE, np.nan)
clinical.head()

## Disease duration features

Longer disease duration of hypertension / diabetes / dyslipidemia is associated with greater brain atrophy, so onset ages are converted to durations: `*_DUR = T_AGE - onset age` for diagnosed participants (clipped at 0 - a few onset ages exceed current age), `0` for participants never diagnosed. Diagnosed participants with an unknown onset age get the median duration of the diagnosed group - this uses predictor data only (no target), so it is not a leakage concern. Unknown disease status stays `NaN` and is imputed in the modeling pipeline.

In [ ]:
for status, onset, name in [("T_HTN", "T_HTNAG", "HTN_DUR"), ("T_DM", "T_DMAG", "DM_DUR"), ("T_LIP", "T_LIPAG", "LIP_DUR")]:
    diseased = clinical[status] == 2  # 1=no, 2=yes
    dur = (clinical["T_AGE"] - clinical[onset]).clip(lower=0)
    dur[clinical[status] == 1] = 0
    n_unknown = (diseased & dur.isna()).sum()
    dur[diseased & dur.isna()] = dur[diseased].median()
    clinical[name] = dur
    print(f"{name}: diagnosed n={diseased.sum()}, onset unknown n={n_unknown}, "
          f"median duration (diagnosed)={dur[diseased].median():.1f}y, NaN={dur.isna().sum()}")

## Select clinical feature set

Curated subset rather than all ~60 raw variables - drops columns that are almost entirely skip-pattern missingness (e.g. individual alcohol-type amount variables like `T_RICEAM`/`T_HLIQAM`, which were almost entirely missing in the EDA) and pregnancy/menopause variables not relevant here. Keeps demographics, disease status + family history, cardiometabolic labs, and summary smoking/alcohol variables.

Added disease durations (`HTN_DUR`, `DM_DUR`, `LIP_DUR`), smoking duration (`T_SMDUYR`, years) and amount (`T_SMAM`, cigarettes/day), drinking duration (`T_DRDU`, ordinal 0=never, 1=<=5y ... 4=21y+) and secondhand smoke (`T_PSM`). `T_SMDUMO` is not used: it is only the months remainder of "X years Y months" and much of it is 66666 (not surveyed). Note `T_PACKYR` = `T_SMDUYR` x `T_SMAM` / 20 (near-exactly), so the two new smoking features recover pack-years with far less missingness.

Then any feature with **>5% true missingness** (66666/99999 only - structural 77777 was recoded above) is dropped. `T_TOTALC` is dropped explicitly as well (by request).

In [ ]:
CLINICAL_FEATURES = [
    "T_ID",
    # demographics
    "T_AGE", "T_SEX", "T_INCOME", "T_MARRY",
    # disease status + duration + family history
    "T_HTN", "T_DM", "T_LIP", "HTN_DUR", "DM_DUR", "LIP_DUR",
    "T_FMFHT1", "T_FMFHT2", "T_FMFDM1", "T_FMFDM2",
    # cardiometabolic labs
    "T_SBP", "T_DBP", "T_PULSE", "T_BMI", "T_HBA1C", "T_GLU0",
    "T_TCHL", "T_HDL", "T_LDL", "T_TG",
    # lifestyle summary
    "T_SMOKE", "T_PACKYR", "T_SMDUYR", "T_SMAM", "T_PSM", "T_DRINK", "T_DRDU", "T_TOTALC", "T_EXER",
]

MAX_MISSING_PCT = 5
EXPLICIT_DROP = ["T_TOTALC"]

missing_pct = clinical[CLINICAL_FEATURES].isna().mean().sort_values(ascending=False) * 100
print("True missingness (%), features with any missing:")
print(missing_pct[missing_pct > 0].round(2).to_string())

dropped = sorted(set(missing_pct[missing_pct > MAX_MISSING_PCT].index) | set(EXPLICIT_DROP))
print("\nDropped:", dropped)
clinical_sel = clinical[[c for c in CLINICAL_FEATURES if c not in dropped]].copy()
print(clinical_sel.shape)
assert (clinical_sel.isna().mean() * 100).max() <= MAX_MISSING_PCT

## APOE feature

`prs_pipeline/apoe_geno.raw` was produced via `plink2 --extract rs429358,rs7412 --export A` (see `build_prs_pipeline.sh` step 2), giving each person's dosage (0/1/2) of whichever allele PLINK labeled A1 at each SNP. Combine the two dosages into a 9-level categorical genotype-group feature (3 x 3, since 2 unphased biallelic SNPs give 9 possible dosage combinations) - labeled descriptively (`geno_group_i_j`), not as real e2/e3/e4, since real allele-to-isoform direction isn't verifiable here (see caveat above).

In [ ]:
apoe_raw = pd.read_csv(f"{STEP_DIR}/prs_pipeline/apoe_geno.raw", sep="\t")
apoe = apoe_raw[["IID", "rs7412_A", "rs429358_C"]].rename(
    columns={"IID": "T_ID", "rs7412_A": "apoe_rs7412_dosage", "rs429358_C": "apoe_rs429358_dosage"}
)
apoe["apoe_geno_group"] = (
    "g" + apoe["apoe_rs429358_dosage"].astype(int).astype(str)
    + "_" + apoe["apoe_rs7412_dosage"].astype(int).astype(str)
)
apoe = apoe[["T_ID", "apoe_geno_group"]]  # dosages dropped: redundant with the group
print(apoe["apoe_geno_group"].value_counts())
apoe.head()

## PRS feature

`prs_pipeline/build_prs_pipeline.sh` runs, per diagnosis (`T_HTN`, `T_DM`, `T_LIP`), a 5-fold out-of-fold GWAS (age+sex covariates, APOE SNPs excluded) against all ~1M array SNPs. The atrophy targets are never used, so the PRS cannot leak the target. Correlated LD-tag SNPs (e.g. `rs1530440_ld1..20`) are deduplicated to one lead SNP per locus, and each fold's held-out individuals are scored with weights estimated only from the other 4 folds. Five variant-selection methods are applied within each fold's discovery set:

| method | rule |
|---|---|
| `p1e-5` | lead p < 1e-5 |
| `p1e-4` | lead p < 1e-4 |
| `p1e-3` | lead p < 1e-3 |
| `fdr05` | Benjamini-Hochberg FDR < 0.05 over all ~998k locus leads |
| `splithalf` | p < 1e-3 in **both** random halves (4,000 each) of the discovery set, same effect sign |

**GWAS model**: linear probability model (per-allele risk difference), not logistic - plink2's logistic `--glm` is ~100x slower here (~13h for the whole pipeline). On chr1 of the fold-0 discovery set the two are compared for SNP ranking, top SNP and effect-sign agreement (`prs_pipeline/compare_lpm_logistic.py`), which is what variant selection and scoring depend on.

**Method choice without test leakage**: for each diagnosis, the method with the highest out-of-fold ROC-AUC for that diagnosis on the **modeling training split only** (same `train_test_split(test_size=0.2, random_state=42)` as the modeling notebook's regression track) is kept. The 20% test split is never used to pick the method, and the atrophy targets are not used at all.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

targets = pd.read_csv(f"{STEP_DIR}/step1_targets.csv")
# modeling notebook does features.merge(targets) with features in clinical row order, then train_test_split
# on row positions - reproduce the same training rows here
order = clinical[["T_ID"]].merge(targets, on="T_ID")
train_idx, test_idx = train_test_split(np.arange(len(order)), test_size=0.2, random_state=42)
train_ids, test_ids = set(order["T_ID"].iloc[train_idx]), set(order["T_ID"].iloc[test_idx])


def disease_auc(prs_df, col, status_col, ids):
    m = prs_df.merge(clinical[["T_ID", status_col]], on="T_ID")
    m = m[m["T_ID"].isin(ids) & m[status_col].notna()]
    return roc_auc_score(m[status_col] == 2, m[col]) if m[col].std() > 0 else np.nan


prs_oof, prs_insample, selection_rows = {}, {}, []
for prs_name, (gdir, status_col) in PRS_DISEASES.items():
    prs_oof[prs_name] = pd.read_csv(f"{STEP_DIR}/{gdir}/prs_all_methods.csv")
    prs_insample[prs_name] = pd.read_csv(f"{STEP_DIR}/{gdir}/prs_insample_all_methods.csv")
    for col in [c for c in prs_oof[prs_name].columns if c != "T_ID"]:
        selection_rows.append({"prs": prs_name, "disease": status_col, "method": col[len(prs_name) + 1:],
                               "train_oof_auc": disease_auc(prs_oof[prs_name], col, status_col, train_ids)})

prs_selection = pd.DataFrame(selection_rows)
prs_selection["chosen"] = prs_selection.groupby("prs")["train_oof_auc"].transform("max") == prs_selection["train_oof_auc"]
prs_selection.to_csv(f"{STEP_DIR}/step1_prs_method_selection.csv", index=False)
best_method = prs_selection[prs_selection["chosen"]].set_index("prs")["method"].to_dict()
print("chosen methods:", best_method)

prs = pd.DataFrame({"T_ID": clinical["T_ID"]})
for prs_name, method in best_method.items():
    prs = prs.merge(prs_oof[prs_name][["T_ID", f"{prs_name}_{method}"]].rename(
        columns={f"{prs_name}_{method}": prs_name}), on="T_ID")
print(prs.shape)
prs_selection.round(4)

In [ ]:
selection_summary = pd.concat([
    pd.read_csv(f"{STEP_DIR}/{gdir}/selection_summary.csv").assign(prs=prs_name)
    for prs_name, (gdir, _) in PRS_DISEASES.items()
])
# out-of-fold folds only; fold == "all" is the in-sample comparison GWAS
selection_summary = selection_summary[selection_summary["fold"] != "all"]
selection_summary.groupby(["prs", "method"])[["n_selected", "n_named_loci"]].agg(["mean", "min", "max"])

In [ ]:
# which named (non-background) loci were selected, and in how many of the 5 folds
named = selection_summary.assign(locus=selection_summary["named_loci"].fillna("").str.split(";")).explode("locus")
named = named[named["locus"] != ""]
named.groupby(["prs", "method", "locus"]).size().unstack("method", fill_value=0)

### Variants selected by `fdr05`

Every locus selected by the `fdr05` rule (named GWAS loci **and** `bg_*` background variants, i.e. all `n_selected` loci), with the number of the 5 out-of-fold discovery sets in which it was selected. Loci are the LD-deduplicated base IDs; `lead_variant` lists which tag was the lowest-p representative in the folds where it was selected (e.g. `rs1530440_ld<k>`). `gene` is the commonly reported nearest/candidate gene of the named GWAS SNP; `bg_*` variants are synthetic background variants with no gene annotation. Saved to `Github/tables/prs_variant.csv`.

In [ ]:
import os

TABLE_DIR = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Github2/tables"
SNP_ARRAY_DIR = "/mnt/d/Healthcare_AI_Challenge/Healthcare_AI_Challenge_2026/SNP_ARRAY"
OOF_FOLDS = [f"fold{k}" for k in range(5)]

# commonly reported nearest/candidate gene per named GWAS SNP (GWAS Catalog / original GWAS reports)
SNP_GENE = {
    # blood pressure / hypertension
    "rs3774372": "ULK4", "rs17367504": "MTHFR-NPPB", "rs880315": "CASZ1", "rs11014166": "CACNB2",
    "rs11191548": "CYP17A1-NT5C2", "rs10757272": "CDKN2A/B (9p21)", "rs16998073": "FGF5",
    "rs7865618": "CDKN2B-AS1 (9p21)", "rs2157719": "CDKN2B-AS1 (9p21)", "rs381815": "PLEKHA7", "rs5068": "NPPA",
    "rs17249754": "ATP2B1", "rs671": "ALDH2", "rs1799998": "CYP11B2", "rs16948048": "ZNF652",
    "rs3184504": "SH2B3", "rs1378942": "CSK", "rs11066280": "HECTD4 (12q24)", "rs1530440": "C10orf107",
    "rs12946454": "PLCD3",
    # type 2 diabetes
    "rs10010131": "WFS1", "rs10811661": "CDKN2A/B (9p21)", "rs10830963": "MTNR1B", "rs10923931": "NOTCH2",
    "rs10946398": "CDKAL1", "rs7754840": "CDKAL1", "rs1111875": "HHEX-IDE", "rs1260326": "GCKR",
    "rs12779790": "CDC123-CAMK1D", "rs13266634": "SLC30A8", "rs163184": "KCNQ1", "rs2237892": "KCNQ1",
    "rs1801282": "PPARG", "rs2233580": "PAX4", "rs4402960": "IGF2BP2", "rs4607103": "ADAMTS9",
    "rs5219": "KCNJ11", "rs7578597": "THADA", "rs7903146": "TCF7L2", "rs7961581": "TSPAN8-LGR5",
    "rs8050136": "FTO", "rs864754": "JAZF1",
    # lipids
    "rs11206510": "PCSK9", "rs12229654": "MYL2-CUX2 (12q24)", "rs12916": "HMGCR", "rs17145738": "MLXIPL",
    "rs1800588": "LIPC", "rs2075291": "APOA5", "rs2266788": "APOA5", "rs651821": "APOA5", "rs662799": "APOA5",
    "rs2131925": "ANGPTL3", "rs2954029": "TRIB1", "rs328": "LPL", "rs3764261": "CETP", "rs708272": "CETP",
    "rs405509": "APOE (promoter)", "rs449647": "APOE (promoter)", "rs769446": "APOE (promoter)",
    "rs4299376": "ABCG5/8", "rs629301": "CELSR2-SORT1", "rs6511720": "LDLR", "rs693": "APOB",
}

variant_rows = []
for prs_name, (gdir, _) in PRS_DISEASES.items():
    for fold in OOF_FOLDS:
        score = pd.read_csv(f"{STEP_DIR}/{gdir}/score_files/score_{fold}.txt", sep="\t")
        sel = score.loc[score["BETA_fdr05"] != 0, "SNP"]
        variant_rows += [{"prs": prs_name, "fold": fold, "variant": v} for v in sel]
variants = pd.DataFrame(variant_rows)
variants["locus"] = variants["variant"].str.replace(r"_ld\d+$", "", regex=True)

# one selected lead per locus per fold, so per-fold counts must match n_selected
n_check = variants.groupby(["prs", "fold"]).size()
n_summary = selection_summary[selection_summary["method"] == "fdr05"].set_index(["prs", "fold"])["n_selected"]
assert n_check.sort_index().equals(n_summary.sort_index()), "variant counts do not match n_selected"

# chromosome of each base locus from the array .bim files (bg_chrN_* encode it in the ID)
bim = pd.concat([pd.read_csv(f"{SNP_ARRAY_DIR}/synthetic_kchip_chr{c}.bim", sep="\t", header=None,
                             usecols=[0, 1], names=["chr", "locus"]) for c in range(1, 23)])
locus_chr = bim.set_index("locus")["chr"]

prs_variant = (
    variants.groupby(["prs", "locus"])
    .agg(lead_variant=("variant", lambda s: ";".join(sorted(s.unique()))),
         n_folds=("fold", "nunique"))
    .rename(columns={"n_folds": "n_folds_selected_fdr05 (of 5)"})
    .reset_index()
)
named_locus = ~prs_variant["locus"].str.startswith("bg_")
unmapped = set(prs_variant.loc[named_locus, "locus"]) - set(SNP_GENE)
assert not unmapped, f"no gene annotation for {unmapped}"
prs_variant["gene"] = prs_variant["locus"].map(SNP_GENE).fillna("- (synthetic background variant)")
prs_variant["chr"] = prs_variant["locus"].map(locus_chr).astype(int)
prs_variant = prs_variant[["prs", "locus", "gene", "chr", "lead_variant", "n_folds_selected_fdr05 (of 5)"]]
prs_variant = prs_variant.sort_values(["prs", "n_folds_selected_fdr05 (of 5)", "chr"], ascending=[True, False, True])

os.makedirs(TABLE_DIR, exist_ok=True)
prs_variant.to_csv(f"{TABLE_DIR}/prs_variant.csv", index=False)
print(prs_variant.groupby("prs")["gene"].apply(lambda g: f"{len(g)} loci, {(~g.str.startswith('-')).sum()} named"))
prs_variant.set_index(["prs", "locus"])

## Out-of-fold vs in-sample PRS

Why the PRS stays out-of-fold even though the atrophy target is never used to build it. An in-sample PRS (GWAS on all 10,000, scoring the same 10,000) is fitted partly to each person's own diagnosis, so it re-encodes the observed `T_HTN`/`T_DM`/`T_LIP` - which are already model features - rather than genetic liability alone.

Three checks, per diagnosis, with the same chosen method for both schemes:
1. **AUC for the diagnosis** on the modeling train split (`auc_train`), test split (`auc_test`) and all 10,000 participants without splitting (`auc_total`). The in-sample AUC is inflated by overfitting; the out-of-fold AUC is what the PRS achieves on people it has never seen, e.g. new participants or the Step 2 follow-up cohort.
2. **Within-status PRS difference**: the in-sample PRS separates cases from controls partly through noise fitted to their labels; this shows up as a larger case-control gap than out-of-fold.
3. **Incremental R^2 for the composite atrophy target** over age + sex + the three diagnoses (OLS, modeling training split). If the in-sample PRS adds more than the out-of-fold PRS, the extra is the echo of observed diagnosis, not genetics.

In [ ]:
import statsmodels.api as sm

TARGET = "target_composite_cont"
base = clinical[["T_ID", "T_AGE", "T_SEX", "T_HTN", "T_DM", "T_LIP"]].merge(targets[["T_ID", TARGET]], on="T_ID")
base_train = base[base["T_ID"].isin(train_ids)].dropna()
X0 = sm.add_constant(base_train[["T_AGE", "T_SEX", "T_HTN", "T_DM", "T_LIP"]].astype(float))
r2_base = sm.OLS(base_train[TARGET], X0).fit().rsquared

all_ids = set(clinical["T_ID"])  # all 10,000, no train/test split
cmp_rows = []
for prs_name, (_, status_col) in PRS_DISEASES.items():
    col = f"{prs_name}_{best_method[prs_name]}"
    for scheme, source in [("out_of_fold", prs_oof), ("in_sample", prs_insample)]:
        p = source[prs_name][["T_ID", col]]
        z = p.assign(z=(p[col] - p[col].mean()) / p[col].std())
        m = base_train.merge(z[["T_ID", "z"]], on="T_ID")
        fit = sm.OLS(m[TARGET], sm.add_constant(m[["T_AGE", "T_SEX", "T_HTN", "T_DM", "T_LIP", "z"]].astype(float))).fit()
        cs = z.merge(clinical[["T_ID", status_col]], on="T_ID").dropna()
        cmp_rows.append({
            "prs": prs_name, "method": best_method[prs_name], "scheme": scheme,
            "auc_train": disease_auc(p, col, status_col, train_ids),
            "auc_test": disease_auc(p, col, status_col, test_ids),
            "auc_total": disease_auc(p, col, status_col, all_ids),
            "case_minus_control_sd": cs.loc[cs[status_col] == 2, "z"].mean() - cs.loc[cs[status_col] == 1, "z"].mean(),
            "composite_beta_adj": fit.params["z"], "composite_p_adj": fit.pvalues["z"],
            "composite_incr_r2": fit.rsquared - r2_base,
        })

prs_scheme_cmp = pd.DataFrame(cmp_rows)
prs_scheme_cmp.to_csv(f"{STEP_DIR}/step1_prs_oof_vs_insample.csv", index=False)
print(f"baseline R^2 (age, sex, HTN, DM, LIP -> {TARGET}, train split): {r2_base:.4f}")
prs_scheme_cmp.round(4)

Out-of-fold PRS (the `out_of_fold` rows) is what goes into the feature matrix below; the in-sample PRS is not used downstream.

## Assemble final feature matrix

In [ ]:
features = clinical_sel.merge(apoe, on="T_ID", how="inner").merge(prs, on="T_ID", how="inner")
assert len(features) == 10000, f"expected 10000 rows, got {len(features)}"
assert features["T_ID"].is_unique
assert not features.columns.str.contains("dosage").any()
for dz, dur in [("T_HTN", "HTN_DUR"), ("T_DM", "DM_DUR"), ("T_LIP", "LIP_DUR")]:
    assert (features[dur].dropna() >= 0).all() and (features.loc[features[dz] == 1, dur] == 0).all()

OUT_PATH = "/mnt/c/Users/sjm57/Desktop/Healthcare_AI_Challenge_2026/Notebook/20261021_Github2/step1_features.csv"
features.to_csv(OUT_PATH, index=False)
print("saved:", OUT_PATH, features.shape)
features.head()